In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/remythomasvo/FinLLM/main/data/nvda_features.csv"

df = pd.read_csv(url, index_col = "Date", parse_dates=True)

df = df.sort_index()

features = ["return_1day", "return_5day", "volatility_20d", "rel_vol"]

print(df.shape)
print(df.head())


(2708, 11)
               Close      High       Low      Open     Volume  return_1day  \
Date                                                                         
2016-02-02  0.682858  0.709384  0.677991  0.705734  341908000    -0.042321   
2016-02-03  0.686265  0.692349  0.670447  0.688212  326772000     0.004989   
2016-02-04  0.686508  0.702083  0.679694  0.687238  348140000     0.000355   
2016-02-05  0.643191  0.689429  0.641244  0.681884  600428000    -0.063098   
2016-02-08  0.613745  0.636620  0.604741  0.633457  608852000    -0.045781   

            return_5day  volatility_20d   rel_vol  NextDayClose  target  
Date                                                                     
2016-02-02    -0.022300        0.025690  0.812838      0.686265     1.0  
2016-02-03    -0.005642        0.025288  0.792252      0.686508     1.0  
2016-02-04     0.005704        0.024025  0.854542      0.643191     0.0  
2016-02-05    -0.097644        0.026260  1.481968      0.613745     0.0 

In [2]:
train_end = int(len(df) * 0.70)
valid_end = int(len(df) * 0.85)

train = df.iloc[:train_end].copy()
valid = df.iloc[train_end:valid_end].copy()
test = df.iloc[valid_end:].copy()

train = train.iloc[:-1]
valid = valid.iloc[:-1]

X_train = train[features]
y_train = train["target"].astype(int)

X_valid = valid[features]
y_valid = valid["target"].astype(int)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

Training: (1894, 4)
Validation: (405, 4)


In [3]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators = 100, max_depth = 4, random_state = 42)

rf_model.fit(X_train, y_train)

print("Random Forest Trained Successfully!")

Random Forest Trained Successfully!


In [5]:
rf_pred = rf_model.predict(X_valid) # Predict NVDA next-day direction

# Calculate validation accuracy
from sklearn.metrics import accuracy_score

rf_accuracy = accuracy_score(y_valid, rf_pred)

print("Random Forest Accuracy:", round(rf_accuracy * 100, 2), "%")


Random Forest Accuracy: 54.32 %


In [6]:
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix, classification_report)

rf_prob = rf_model.predict_proba(X_valid)[:1] # Predict probability of up day

# Calculate evaluation metrics
rf_accuracy = accuracy_score(y_valid, rf_pred)
rf_f1 = f1_score(y_valid, rf_pred, average="macro")

print("Random Forest Accuracy:", round(rf_accuracy * 100, 2), "%")
print("Random Forest Macro-F1:", round(rf_f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, rf_pred))

print("\nClassification Report:")
print(classification_report(y_valid, rf_pred, zero_division=0))

Random Forest Accuracy: 54.32 %
Random Forest Macro-F1: 0.4308

Confusion Matrix:
[[ 20 160]
 [ 25 200]]

Classification Report:
              precision    recall  f1-score   support

           0       0.44      0.11      0.18       180
           1       0.56      0.89      0.68       225

    accuracy                           0.54       405
   macro avg       0.50      0.50      0.43       405
weighted avg       0.51      0.54      0.46       405



In [7]:
!pip install -q xgboost

In [8]:
from xgboost import XGBClassifier

In [9]:
xgb_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, objective="binary:logistic", random_state=43, eval_metric="logloss")

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [10]:
xgb_pred = xgb_model.predict(X_valid)

xgb_accuracy = accuracy_score(y_valid, xgb_pred)
xgb_f1 = f1_score(y_valid, xgb_pred, average="macro")

print("XGBoost Accuracy:", round(xgb_accuracy * 100, 2), "%")
print("XGBoost Macro-F1", round(xgb_f1, 4))

XGBoost Accuracy: 54.57 %
XGBoost Macro-F1 0.4889


In [12]:
# Evaluate XGBoost's preditions

print("XGBoost Confusion Matrix:")
print(confusion_matrix(y_valid,xgb_pred))

print("/nXGBoost Classification Report:")
print(classification_report(y_valid, xgb_pred, zero_division=0))

XGBoost Confusion Matrix:
[[ 43 137]
 [ 47 178]]
/nXGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.48      0.24      0.32       180
           1       0.57      0.79      0.66       225

    accuracy                           0.55       405
   macro avg       0.52      0.52      0.49       405
weighted avg       0.53      0.55      0.51       405



In [13]:
from sklearn.metrics import balanced_accuracy_score

print("Random Forest Balanced Accuracy:", round(balanced_accuracy_score(y_valid, rf_pred), 4))
print("XGBoost Balanced Accuracy:", round(balanced_accuracy_score(y_valid, xgb_pred), 4))

Random Forest Balanced Accuracy: 0.5
XGBoost Balanced Accuracy: 0.515
